# Automatización Fase 3 (Búsqueda de vuelos) — asignar Instructor + Actividad

Continúa donde quedaron los otros dos notebooks:
- `Automatizacion_Fase1_2_Demanda_Instructores.ipynb` → escribió `"LCK A320F"` en la
  Matriz (Rol de Instructores), reservando qué instructor cubre qué día.
- `Automatizacion_LCK_A320F_NB.ipynb` → arma los bloques candidatos (2 pairings = 1 día de
  instructor) desde BigQuery, con Instructor/Actividad en blanco (dropdown) para llenar
  a mano o desde otra hoja.

**Este notebook cierra ese hueco**: lee la Matriz real (ya con los slots reservados),
busca -para cada (instructor, fecha) reservado- un bloque candidato de ESA fecha exacta, y
recién ahí escribe el nombre del instructor y la actividad `"LCK A320F"` en el Excel de
bloques. **No modifica** `generar_candidatos_nb.py` ni `generar_reporte_pairings_nb.py`
(siguen intactos); acá se usa una copia de `construir_bloques_nb` con un parámetro nuevo
(`asignaciones_por_bloque`) que solo agrega valores a celdas que ya existían en blanco —
no cambia el formato ni las reglas de negocio ya validadas.

**Decisiones confirmadas con Fernando para este paso (no inventadas):**
- Si para un (instructor, fecha) reservado NO hay un bloque completo (2 pairings, 8 TC) en
  esa fecha exacta, **se deja sin asignar y se reporta aparte** — no se busca una fecha
  alternativa ni se relaja ninguna regla ya validada (conexión, PSV, HBT, etc.).
- El nombre del instructor se matchea entre la Matriz y el catálogo de `Instructores` del
  archivo de Pairings **normalizando tildes/mayúsculas**; si no hay match, se reporta en vez
  de adivinar o crear un instructor nuevo.
- La actividad se llena siempre como `"LCK A320F"` (es lo único que calcula la demanda de
  Archivo 9 en la Fase 1) — otras actividades (auditorías, habilitación, etc.) siguen
  siendo manuales, igual que antes.


## 1. Instalar dependencias y autenticar

In [ ]:
!pip install -q gspread openpyxl


In [ ]:
import sys
import re
import unicodedata
import collections
from collections import defaultdict
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
import gspread
from google.auth import default
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.comments import Comment
from openpyxl.worksheet.datavalidation import DataValidation

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
# Proyecto de FACTURACIÓN (donde se corre/paga el job) -> distinto del proyecto
# DUEÑO de los datos (operations-data-prod, referenciado en el FROM de la query).
# La cuenta de Fernando no tiene "bigquery.jobs.create" en operations-data-prod
# (error 403 real) pero sí lo tiene en datadem-home.
client = bigquery.Client(project="datadem-home")
print("Autenticado: BigQuery + Google Sheets con tu cuenta.")


## 2. Query a BigQuery + armado de candidatos/bloques (Fase 0 + Fase 3 pasos 14-19)

Copia literal (sin cambios) de las mismas funciones ya validadas en
`Automatizacion_LCK_A320F_NB.ipynb` — mismas reglas 2.9-2.14, mismo `separar_instancias_trip`,
mismo `parear_candidatos`. Si ese notebook ya corrió este mes, este bloque va a dar
exactamente los mismos bloques.

In [ ]:
MES_OBJETIVO = 10
ANIO_OBJETIVO = 2026

WD_ES = {
    "Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles",
    "Thursday": "jueves", "Friday": "viernes", "Saturday": "sábado",
    "Sunday": "domingo",
}

RUTAS_VALIDAS_ARR = {"AQP", "CIX", "CJA", "CUZ", "PEM", "PIU", "TBP", "TPP", "TCQ"}
# Confirmado con Fernando: AQP sigue excluido en octubre-2026 también (no solo sept-2026).
# Revisar cada mes si sigue vigente -> no asumir que se repite indefinidamente.
EXCLUSIONES_MES = {"AQP"}

HORA_MIN_SALIDA = pd.Timedelta(hours=8, minutes=30)
HBT_MIN = pd.Timedelta(hours=1)
CONEXION_MIN = pd.Timedelta(minutes=50)
CONEXION_MAX = pd.Timedelta(hours=1, minutes=30)
PSV_MAX = pd.Timedelta(hours=11)

QUERY_NB = """
SELECT
  pairing_id                       AS trip,
  flight_start_date_local_time     AS inicio_vuelo_lt,
  duty_calendar_day_number         AS dia_duty,
  flight_number                    AS vuelo,
  departure_airport_code           AS dep,
  arrival_airport_code             AS arr,
  flight_departure_time_crew_base  AS std_hb,
  flight_arrival_hour_block_time   AS sta_hb,
  flight_block_time                AS hbt,
  subfleet_code                    AS sub_fleet

FROM `operations-data-prod.carmen_gold.crew_pairing_carmen_system`

WHERE
  flight_start_date_local_time BETWEEN DATE '2025-10-01' AND DATE '2026-10-31'
  AND subsidiary_code IN ('LP')
  AND load_type_code = 'FP'
  AND crew_range_type_code = 'SAB'
  AND subfleet_code IN ('319', '320')

QUALIFY
  CASE
    WHEN load_type_code = 'FP' AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'FP' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    WHEN load_type_code = 'ES' AND
         MAX(CASE WHEN load_type_code = 'FP' THEN 0 ELSE 0 END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year) = -1 AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'ES' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    ELSE -1
  END = 0

ORDER BY pairing_id ASC
"""

# create_bqstorage_client=False: evita la BigQuery Storage API (pide el
# permiso aparte "bigquery.readsessions.create", que la cuenta de Fernando
# no tiene en datadem-home -> error 403 real). Sin ese flag, to_dataframe()
# intenta usarla automáticamente aunque la query en sí ya haya corrido bien.
df_raw = client.query(QUERY_NB).to_dataframe(create_bqstorage_client=False)
print(f"Filas descargadas: {len(df_raw)}")

In [ ]:
def parse_hora(s):
    if pd.isna(s):
        return pd.NaT
    h, m, sec = str(s).split(":")
    sec = sec.split(".")[0]
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def cargar_bq_a_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    df["trip"] = df["trip"].astype(str)
    df["dia_duty"] = df["dia_duty"].astype(int)
    df["fecha_dt"] = pd.to_datetime(df["inicio_vuelo_lt"])
    df["std_td"] = df["std_hb"].apply(parse_hora)
    df["sta_td"] = df["sta_hb"].apply(parse_hora)
    df["hbt_td"] = df["hbt"].apply(parse_hora)
    df["std_dt"] = df["fecha_dt"] + df["std_td"]
    df["sta_dt"] = df["fecha_dt"] + df["sta_td"]
    df.loc[df["sta_dt"] < df["std_dt"], "sta_dt"] += pd.Timedelta(days=1)
    df["dia_semana"] = df["fecha_dt"].dt.day_name().map(WD_ES)
    return df


def separar_instancias_trip(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["trip", "fecha_dt", "std_dt"]).reset_index(drop=True)
    instancia = []
    trip_actual = None
    max_dia_duty = -1
    idx_instancia = 0
    for _, row in df.iterrows():
        if row["trip"] != trip_actual:
            trip_actual = row["trip"]
            idx_instancia = 0
            max_dia_duty = row["dia_duty"]
        elif row["dia_duty"] < max_dia_duty:
            idx_instancia += 1
            max_dia_duty = row["dia_duty"]
        else:
            max_dia_duty = max(max_dia_duty, row["dia_duty"])
        trip_val = row["trip"]
        instancia.append(f"{trip_val}_{idx_instancia}")

    df["trip_original"] = df["trip"]
    df["trip"] = instancia
    return df


def filtrar_mes_y_ruta(df: pd.DataFrame, mes: int, anio: int) -> pd.DataFrame:
    rutas_validas = RUTAS_VALIDAS_ARR - {c.upper() for c in EXCLUSIONES_MES}
    en_mes = (df["fecha_dt"].dt.month == mes) & (df["fecha_dt"].dt.year == anio)
    sale_de_lim = df["dep"] == "LIM"
    llega_a_lim = df["arr"] == "LIM"
    ruta_nacional_ok = df["arr"].isin(rutas_validas) | (llega_a_lim)
    return df[en_mes & (sale_de_lim | llega_a_lim) & ruta_nacional_ok].copy()


def armar_primeras_mitades(df: pd.DataFrame, dia_duty_min_real=None):
    validos_rows = []
    excluidos_rows = []

    for trip, grupo in df.groupby("trip"):
        dia_min = grupo["dia_duty"].min()
        if dia_duty_min_real is not None and trip in dia_duty_min_real.index and dia_min != dia_duty_min_real.loc[trip]:
            excluidos_rows.append({"trip": trip, "motivo": "día 1 real fuera de mes/ruta"})
            continue
        dia1 = grupo[grupo["dia_duty"] == dia_min].sort_values("std_dt")

        if len(dia1) < 2:
            excluidos_rows.append({"trip": trip, "motivo": "día 1 sin vuelta el mismo día"})
            continue
        if len(dia1) in (6, 8, 10):
            excluidos_rows.append({"trip": trip, "motivo": f"día 1 tiene {len(dia1)} tramos"})
            continue

        ida, vuelta = dia1.iloc[0], dia1.iloc[1]
        if ida["dep"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el primer tramo no sale de LIM"})
            continue
        if vuelta["arr"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el segundo tramo no vuelve a LIM"})
            continue
        if vuelta["dep"] != ida["arr"]:
            excluidos_rows.append({"trip": trip, "motivo": "ruta triangular"})
            continue

        motivos = []
        if (ida["std_dt"] - ida["fecha_dt"]) <= HORA_MIN_SALIDA:
            motivos.append("sale antes/igual a 08:30")
        if ida["hbt_td"] <= HBT_MIN:
            motivos.append("HBT ida <= 1h")
        if vuelta["hbt_td"] <= HBT_MIN:
            motivos.append("HBT vuelta <= 1h")
        conexion = vuelta["std_dt"] - ida["sta_dt"]
        if conexion <= pd.Timedelta(0):
            motivos.append("conexión interna negativa/cero")
        psv = vuelta["sta_dt"] - ida["std_dt"]
        if psv > PSV_MAX:
            motivos.append(f"PSV {psv} > 11h")

        if motivos:
            excluidos_rows.append({"trip": trip, "motivo": "; ".join(motivos)})
            continue

        validos_rows.append({
            "Fecha": ida["fecha_dt"].strftime("%d/%m/%Y"),
            "DíaSem": ida["dia_semana"],
            "Pairing ID": trip,
            "Vuelo Ida": ida["vuelo"], "Dep": ida["dep"], "Arr": ida["arr"],
            "STD Ida": ida["std_hb"], "STA Ida": ida["sta_hb"], "HBT Ida": ida["hbt"],
            "Vuelo Vuelta": vuelta["vuelo"], "Dep Vta": vuelta["dep"], "Arr Vta": vuelta["arr"],
            "STD Vuelta": vuelta["std_hb"], "STA Vuelta": vuelta["sta_hb"], "HBT Vuelta": vuelta["hbt"],
            "Conexión": str(conexion), "PSV Total": str(psv),
            "Sub Flota": ida["sub_fleet"],
            "_orden": ida["std_dt"],
        })

    cols_finales = ["Fecha", "DíaSem", "Pairing ID", "Vuelo Ida", "Dep", "Arr", "STD Ida",
                     "STA Ida", "HBT Ida", "Vuelo Vuelta", "Dep Vta", "Arr Vta", "STD Vuelta",
                     "STA Vuelta", "HBT Vuelta", "Conexión", "PSV Total", "Sub Flota"]

    if not validos_rows:
        return pd.DataFrame(columns=cols_finales), pd.DataFrame(excluidos_rows)

    validos = pd.DataFrame(validos_rows).sort_values("_orden")
    validos = validos[cols_finales].reset_index(drop=True)
    return validos, pd.DataFrame(excluidos_rows)


def _fecha_dt(s):
    d, m, a = s.split("/")
    return pd.Timestamp(year=int(a), month=int(m), day=int(d))


def _hora_td(s):
    # BigQuery TIME llega como datetime.time (no string) cuando se usa
    # to_dataframe(create_bqstorage_client=False) -> str(s) lo normaliza
    # a "HH:MM:SS" antes de partirlo, igual que ya hace parse_hora().
    h, m, sec = str(s).split(":")
    sec = sec.split(".")[0]
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def parear_candidatos(validos: pd.DataFrame):
    df = validos.copy()
    df["_ida_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STD Ida"]), axis=1)
    df["_vta_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STA Vuelta"]), axis=1)
    df = df.sort_values("_ida_dt").reset_index(drop=True)

    usados = set()
    bloques = []
    for i, row in df.iterrows():
        if row["Pairing ID"] in usados:
            continue
        ventana_ini = row["_vta_dt"] + CONEXION_MIN
        ventana_fin = row["_vta_dt"] + CONEXION_MAX
        mismo_dia = df[
            (~df["Pairing ID"].isin(usados)) &
            (df["Pairing ID"] != row["Pairing ID"]) &
            (df["Fecha"] == row["Fecha"]) &
            (df["_ida_dt"] > ventana_ini) & (df["_ida_dt"] < ventana_fin) &
            (df["_vta_dt"] - row["_ida_dt"] <= PSV_MAX)
        ].sort_values("_ida_dt")

        usados.add(row["Pairing ID"])
        if len(mismo_dia) > 0:
            segunda = mismo_dia.iloc[0]
            usados.add(segunda["Pairing ID"])
            bloques.append((row, segunda))
        else:
            bloques.append((row, None))
    return bloques


df = cargar_bq_a_df(df_raw)
df = separar_instancias_trip(df)
dia_duty_min_real = df.groupby("trip")["dia_duty"].min()
df_filtrado = filtrar_mes_y_ruta(df, MES_OBJETIVO, ANIO_OBJETIVO)
validos, excluidos = armar_primeras_mitades(df_filtrado, dia_duty_min_real)
bloques = parear_candidatos(validos)

n_parejas = sum(1 for _, b in bloques if b is not None)
n_solos = sum(1 for _, b in bloques if b is None)
print(f"Candidatos válidos: {len(validos)}")
print(f"Bloques armados: {len(bloques)} ({n_parejas} completos, {n_solos} solos)")


## 3. Leer la Matriz real: qué (instructor, fecha) quedaron reservados con "LCK A320F"

Misma estructura ya usada en `Automatizacion_Fase1_2_Demanda_Instructores.ipynb`
(fila 2 = fechas desde columna C, fila 3+ = instructores). Ajusta si tu matriz cambió.

In [ ]:
URL_MATRIZ = "https://docs.google.com/spreadsheets/d/19WmwaoLDZnArNztu_dJNwi7bjrGq-_cx_gw0aN96zfk/edit?gid=580414308"

sh_matriz = gc.open_by_url(URL_MATRIZ)
ws_matriz = sh_matriz.get_worksheet_by_id(580414308)

FILA_ENCABEZADO_FECHAS = 2
FILA_PRIMER_INSTRUCTOR = 3
COL_PRIMERA_FECHA = 3

valores_m = ws_matriz.get_all_values()

fila_fechas = valores_m[FILA_ENCABEZADO_FECHAS - 1]
fechas_matriz = []
for celda in fila_fechas[COL_PRIMERA_FECHA - 1:]:
    fechas_matriz.append(celda.strip() if celda.strip() else None)

reservas = []  # (bp, nombre_matriz, fecha_str "dd/mm/yyyy")
for fila in valores_m[FILA_PRIMER_INSTRUCTOR - 1:]:
    if not fila or not fila[0].strip():
        continue
    bp = fila[0].strip().lstrip("'")
    nombre = fila[1].strip() if len(fila) > 1 else ""
    for j, celda in enumerate(fila[COL_PRIMERA_FECHA - 1:]):
        if celda.strip().upper() == "LCK A320F":
            fecha_str = fechas_matriz[j] if j < len(fechas_matriz) else None
            if fecha_str:
                reservas.append((bp, nombre, fecha_str))

print(f"Slots 'LCK A320F' encontrados en la Matriz: {len(reservas)}")


## 4. Emparejar cada reserva de la Matriz con un bloque candidato de esa fecha

Reglas confirmadas (sin inventar):
- Solo se asigna si hay un bloque **completo** (2 pairings) en la fecha EXACTA reservada.
- Si no hay bloque completo esa fecha, se reporta en `sin_bloque` (no se busca fecha
  alternativa ni se usa un bloque "solo" de 1 pairing).
- El nombre se matchea normalizando tildes/mayúsculas contra el catálogo `INSTRUCTORES_DATA`;
  si no hay match, se reporta en `sin_match_nombre` (no se inventa ni se crea instructor).

In [ ]:
INSTRUCTORES_DATA = [
    ("Christian Rondon", "1271571", " RONDON BARRUTIA CHRISTIAN ERIC "),
    ("Erika Davila", "967092", "DAVILA BELLO MARIA ERIKA"),
    ("Sebastian Correa", "2396710", " CORREA GARCIA JUAN SEBASTIAN "),
    ("Fiorella Ruiz", "2713993", "RUIZ RIOJA FIORELLA DEL PILAR"),
    ("Jazmin Guerra", "29530", "GUERRA SUAREZ JAZMIN"),
    ("Jennifert Acurio", "3779550", "ACURIO DARGENT JENNIFERT MILAGROS"),
    ("Karen Santa Cruz", "2843319", "SANTA CRUZ HUAMAN KAREN"),
    ("Luis Bacigalupo", "2963161", "BACIGALUPO FLORES LUIS ENRIQUE"),
    ("Claudia Flores", "3217561", " FLORES FUENTES DAVILA CLAUDIA ALEXANDRA "),
    ("Karla Moz", "71348", "MOZ MONTES KARLA LISSETTE"),
    ("Elizabeth Torres", "2369641", "TORRES POLO ELIZABETH DEL PILAR"),
    ("Patricia Najar", "2369624", "NAJAR CRUZ PATRICIA DEL PILAR"),
    ("Javier Zapata", "3134911", "ZAPATA GARAYAR JAVIER RICARDO SALVADOR"),
    ("Jefferson Mendez", "3750335", " MENDEZ RUCOBA JEFFERSON "),
    ("Gabriela Ungaro", "3852423", "UNGARO GUTIERREZ GABRIELA"),
    ("Mariella Carrasco", "2604360", "CARRASCO BENAVIDES ROSA MARIELLA"),
    ("Cesar Campos", "2823133", " CAMPOS CONCHE CESAR AUGUSTO "),
    ("Kevin Segovia", "3189967", "SEGOVIA TAPIA RAY KEVIN"),
    ("Milagros Salas", "2415373", "SALAS COSIO MILAGROS PATRICIA"),
    ("Judith Fernandez", "2440915", "FERNANDEZ GARCIA JUDITH JULIET"),
    ("Rafael Nieto", "3796947", " NIETO SAENZ RAFAEL ANTONIO "),
    ("Gianfranco Celiz", "3841387", " CELIZ ROSSI GIANFRANCO PAOLO "),
]


def normalizar_nombre(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.strip().lower()


def normalizar_fecha(fecha_str):
    # La Matriz muestra las fechas sin cero adelante (ej. "1/10/2026"),
    # pero los bloques usan strftime("%d/%m/%Y") -> "01/10/2026". Comparar
    # como texto plano hacia fallar el match en TODOS los días 1-9 del mes
    # (bug real encontrado: 9 de 24 reservas quedaban "sin bloque" aunque
    # el bloque completo sí existía). Se normaliza a (día, mes, año) como
    # enteros para que "1/10/2026" y "01/10/2026" sean la misma fecha.
    d, m, a = fecha_str.strip().split("/")
    return (int(d), int(m), int(a))


def emparejar_matriz_con_bloques(reservas, bloques):
    mapa_nombre_canonico = {normalizar_nombre(n): n for n, _bp, _legal in INSTRUCTORES_DATA}

    bloques_por_fecha = defaultdict(list)
    for idx, (candA, _candB) in enumerate(bloques):
        bloques_por_fecha[normalizar_fecha(candA["Fecha"])].append(idx)

    usados_bloques = set()
    asignaciones_por_bloque = {}
    sin_bloque = []
    sin_match_nombre = []

    for bp, nombre_matriz, fecha_str in reservas:
        nombre_canonico = mapa_nombre_canonico.get(normalizar_nombre(nombre_matriz))
        if nombre_canonico is None:
            sin_match_nombre.append((bp, nombre_matriz, fecha_str))
            continue

        fecha_norm = normalizar_fecha(fecha_str)
        candidatos_idx = [i for i in bloques_por_fecha.get(fecha_norm, []) if i not in usados_bloques]
        candidatos_completos = [i for i in candidatos_idx if bloques[i][1] is not None]

        elegido = candidatos_completos[0] if candidatos_completos else None
        if elegido is None:
            sin_bloque.append((bp, nombre_canonico, fecha_str))
            continue

        usados_bloques.add(elegido)
        asignaciones_por_bloque[elegido] = (nombre_canonico, "LCK A320F")

    return asignaciones_por_bloque, sin_bloque, sin_match_nombre


asignaciones_por_bloque, sin_bloque, sin_match_nombre = emparejar_matriz_con_bloques(reservas, bloques)

print(f"Reservas de la Matriz: {len(reservas)}")
print(f"Asignadas a un bloque: {len(asignaciones_por_bloque)}")
print(f"Sin bloque completo en esa fecha exacta (revisar a mano): {len(sin_bloque)}")
for bp, nombre, fecha in sin_bloque:
    print(f"  - {nombre} (BP {bp}) reservado el {fecha}, no hay bloque completo ese día")
print(f"Nombres sin match en el catálogo de Instructores (revisar a mano): {len(sin_match_nombre)}")
for bp, nombre, fecha in sin_match_nombre:
    print(f"  - '{nombre}' (BP {bp}, {fecha}) no matchea ningún instructor del catálogo")


## 5. Armar el Excel de bloques, ya con Instructor + Actividad completados

Copia de `construir_bloques_nb` con UN solo agregado: el parámetro `asignaciones_por_bloque`
(dict `{índice_de_bloque: (nombre_instructor, actividad)}`). Si un bloque no está en el dict,
queda exactamente igual que antes (celda en blanco con su dropdown) — no cambia nada del
formato ni de las reglas ya validadas.

In [ ]:
ACTIVIDADES_NB = [
    "LCK A320F", "Reentrenamiento A320F", "LCK A320F + Habilitación A320F",
    "LCK A320 ALUMNO", "Auditoria CAB Vuelo", "BIANUAL IDE - HAB IDE",
    "EXP RECIENTE A320", "CHEQUEO LATAM A320F", "CHEQUEO DGAC A320F",
]

TITULOS_GRID = [
    (1, 2, "Conexión > o = a 50min y < a 1hr y 30min", False),
    (1, 4, "NO CONSIDERAR TRU/JUL/JAE/AYP/JAU/IQT", True),
    (1, 8, "CONSIDERAR PAIRINGS PARTIDOS LCK A320", False),
    (1, 11, "Vuelos LCK = Siempre en Flota 320", False),
    (2, 2, "Vuelos HBT mayor a 1 hora", False),
    (2, 4, "PSV NO MAYOR A 11 HRS", False),
    (2, 8, "CONSIDERAR SIEMPRE EL PDR", False),
    (2, 11, "Vuelos iniciando más de 08:30", False),
    (3, 4, "NO REPETIR PAIRING EN LCK", True),
]

HEADERS = ["Pairing ID", "Fecha", "DíaSem", "Vuelo", "Dep", "Arr", "STD", "STA", "Sub Flota"]
COLUMNAS_HORA = {"STD", "STA"}  # forzar formato texto: ver nota en el bucle de escritura


def construir_bloques_nb(bloques, out_path, n_muestra=None, asignaciones_por_bloque=None):
    asignaciones_por_bloque = asignaciones_por_bloque or {}
    wb = Workbook()
    ws = wb.active
    ws.title = "Pairings NB"

    bold = Font(bold=True)
    red_bold = Font(bold=True, color="CC0000")
    green_bold = Font(bold=True)
    dutyid_fill = PatternFill("solid", fgColor="FFC000")
    psv_fill = PatternFill("solid", fgColor="FFFF00")
    resumen_fill = PatternFill("solid", fgColor="D9E1F2")
    conexion_lim_fill = PatternFill("solid", fgColor="C6E8C6")
    thin = Side(style="thin", color="999999")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center", wrap_text=True)

    COL_CONEXION, COL_HBT = 11, 12
    COL_INS, COL_VACIA1, COL_DESC = 13, 14, 15
    COL_VACIA2 = 16
    COL_RESUMEN_INI = 17
    RESUMEN_HEADERS = ["Pairing ID", "Fecha", "DíaSem", "Vuelo", "Ruta",
                        "Instructor", "Actividad", "Sub Flota"]

    wi = wb.create_sheet("Instructores")
    for j, h in enumerate(["Instructor", "BP", "Nombre"]):
        c = wi.cell(row=1, column=1 + j, value=h)
        c.font = bold
        c.border = border
    for i, (nombre, bp, legal) in enumerate(INSTRUCTORES_DATA, start=2):
        wi.cell(row=i, column=1, value=nombre).border = border
        wi.cell(row=i, column=2, value=bp).border = border
        wi.cell(row=i, column=3, value=legal).border = border
    for c, w in zip("ABC", (18, 12, 40)):
        wi.column_dimensions[c].width = w
    n_instructores = len(INSTRUCTORES_DATA)

    dv_instructor = DataValidation(
        type="list", formula1=f"=Instructores!$A$2:$A${1 + n_instructores}", allow_blank=True)
    ws.add_data_validation(dv_instructor)

    for fila_r, col_r, texto, es_rojo in TITULOS_GRID:
        c = ws.cell(row=fila_r, column=col_r, value=texto)
        c.font = red_bold if es_rojo else bold

    fila_labels = 6

    for j, h in enumerate(RESUMEN_HEADERS):
        c = ws.cell(row=fila_labels, column=COL_RESUMEN_INI + j, value=h)
        c.font = bold
        c.fill = resumen_fill
        c.border = border
        c.alignment = center

    dv = DataValidation(type="list", formula1='"' + ",".join(ACTIVIDADES_NB) + '"', allow_blank=True)
    ws.add_data_validation(dv)

    todas_filas_pairing = []
    HEADER_TITULOS = ["Pairing ID", "FECHA REAL", "Day of Week", "Flight No",
                       "Dep Stn", "Arr Stn", "STD", "STA", "subflota", "Conexion", "HBT"]

    fila = fila_labels + 2
    if n_muestra is not None:
        bloques = bloques[:n_muestra]

    for idx_bloque, (candA, candB) in enumerate(bloques):
        candidatos_bloque = [candA] + ([candB] if candB is not None else [])
        filas_bloque = []

        for j, titulo in enumerate(HEADER_TITULOS):
            c = ws.cell(row=fila, column=2 + j, value=titulo)
            c.font = bold
            c.fill = resumen_fill
            c.border = border
            c.alignment = center
        fila += 1

        fila_r1 = fila
        for cand in candidatos_bloque:
            r_ida, r_vta = fila, fila + 1
            # STD/STA se fuerzan a texto: BigQuery (create_bqstorage_client=False)
            # devuelve datetime.time, no string. Si se escribe tal cual, openpyxl
            # lo guarda como HORA NUMÉRICA real -> las fórmulas TIMEVALUE(H..) y
            # LEFT(H..,5) de más abajo esperan TEXTO ("13:25:00") y tiran #VALOR!
            # con un valor numérico. Bug real encontrado en la hoja generada.
            ida = {"Pairing ID": cand["Pairing ID"], "Fecha": cand["Fecha"], "DíaSem": cand["DíaSem"],
                   "Vuelo": cand["Vuelo Ida"], "Dep": cand["Dep"], "Arr": cand["Arr"],
                   "STD": str(cand["STD Ida"]), "STA": str(cand["STA Ida"]), "Sub Flota": cand["Sub Flota"]}
            vta = {"Pairing ID": cand["Pairing ID"], "Fecha": cand["Fecha"], "DíaSem": cand["DíaSem"],
                   "Vuelo": cand["Vuelo Vuelta"], "Dep": cand["Dep Vta"], "Arr": cand["Arr Vta"],
                   "STD": str(cand["STD Vuelta"]), "STA": str(cand["STA Vuelta"]), "Sub Flota": cand["Sub Flota"]}

            for r, leg in ((r_ida, ida), (r_vta, vta)):
                for j, h in enumerate(HEADERS):
                    c = ws.cell(row=r, column=2 + j, value=leg[h])
                    c.border = border
                    c.alignment = center
                    if h in COLUMNAS_HORA:
                        # Formato de celda = Texto ("@"): sin esto, al hacer
                        # clic + Enter en la celda (sin cambiar nada), Excel
                        # reinterpreta "13:25:00" como hora real y la vuelve
                        # a convertir a numero -> rompe TIMEVALUE()/LEFT() de
                        # nuevo. Bug real reportado por Fernando tras editar
                        # una celda manualmente en el Excel ya generado.
                        c.number_format = "@"
                c = ws.cell(row=r, column=COL_HBT,
                            value=str(cand["HBT Ida"] if r == r_ida else cand["HBT Vuelta"]))
                c.border = border
                c.alignment = center
                c.number_format = "@"

            if r_ida != fila_r1:
                c_con = ws.cell(row=r_ida, column=COL_CONEXION,
                                 value=(f'=TEXT(TIMEVALUE(H{r_ida})-TIMEVALUE(I{r_ida - 1})'
                                        f'+(TIMEVALUE(H{r_ida})<TIMEVALUE(I{r_ida - 1})),"[h]:mm")'))
                c_con.fill = conexion_lim_fill
                c_con.font = green_bold
            ws.cell(row=r_vta, column=COL_CONEXION,
                    value=(f'=TEXT(TIMEVALUE(H{r_vta})-TIMEVALUE(I{r_ida})'
                           f'+(TIMEVALUE(H{r_vta})<TIMEVALUE(I{r_ida})),"[h]:mm")'))

            filas_bloque.append((r_ida, r_vta))
            fila = r_vta + 1

        r1 = filas_bloque[0][0]
        asignacion = asignaciones_por_bloque.get(idx_bloque)

        c_ins = ws.cell(row=r1, column=COL_INS)
        c_ins.fill = dutyid_fill
        c_ins.font = bold
        c_ins.comment = Comment("Elegir el instructor de la lista", "Automatizacion_Fase3")
        dv_instructor.add(c_ins)
        if asignacion is not None:
            c_ins.value = asignacion[0]
        for idx, (r_ida, r_vta) in enumerate(filas_bloque):
            if idx > 0:
                ws.cell(row=r_ida, column=COL_INS, value=f'=IF($M${r1}="","",$M${r1})')

        c_act = ws.cell(row=r1, column=COL_DESC)
        c_act.fill = dutyid_fill
        c_act.font = bold
        c_act.comment = Comment("Tipo de actividad (elegir de la lista)", "Automatizacion_Fase3")
        dv.add(c_act)
        if asignacion is not None:
            c_act.value = asignacion[1]

        for k, (r_ida, r_vta) in enumerate(filas_bloque):
            if k > 0:
                ws.cell(row=r_ida, column=COL_DESC, value=f'=IF($O${r1}="","",$O${r1})')
            ws.cell(row=r_vta, column=COL_DESC,
                    value=(f'="LIM-"&G{r_ida}&"-LIM   LA "&E{r_ida}&" ("&LEFT(H{r_ida},5)&"-"&LEFT(I{r_ida},5)&" hrs)"'
                           f'  /  LA "&E{r_vta}&" ("&LEFT(H{r_vta},5)&"-"&LEFT(I{r_vta},5)&" hrs)"'))

        r_ult = filas_bloque[-1][1]
        fila_psv = fila
        c_psv_label = ws.cell(row=fila_psv, column=10, value="PSV total:")
        c_psv_label.font = bold
        c_psv_label.fill = psv_fill
        c_psv_val = ws.cell(row=fila_psv, column=11,
                             value=f'=TEXT(TIMEVALUE(I{r_ult})-TIMEVALUE(H{r1})+(I{r_ult}<H{r1}),"[h]:mm")')
        c_psv_val.fill = psv_fill

        for r_ida, r_vta in filas_bloque:
            todas_filas_pairing.append((r_ida, r_vta, r1))
            resumen_valores = [
                f"=B{r_ida}", f"=C{r_ida}", f"=D{r_ida}",
                f'=E{r_ida}&"/"&E{r_vta}',
                f'=F{r_ida}&"-"&G{r_ida}&"-"&F{r_ida}',
                f'=IF($M${r1}="","",$M${r1})',
                f'=IF($O${r1}="","",$O${r1})',
                f"=J{r_ida}",
            ]
            for j, val in enumerate(resumen_valores):
                c = ws.cell(row=r_ida, column=COL_RESUMEN_INI + j, value=val)
                c.border = border
                c.alignment = center

        fila = fila_psv + 2

    anchos = [10, 11, 9, 8, 7, 7, 9, 9, 9]
    for j, w in enumerate(anchos):
        ws.column_dimensions[ws.cell(row=1, column=2 + j).column_letter].width = w
    for col, w in ((COL_CONEXION, 9), (COL_HBT, 8), (COL_INS, 18), (COL_VACIA1, 3),
                   (COL_DESC, 38), (COL_VACIA2, 3)):
        ws.column_dimensions[ws.cell(row=1, column=col).column_letter].width = w
    for j, w in enumerate([10, 11, 9, 12, 16, 18, 24, 9]):
        ws.column_dimensions[ws.cell(row=1, column=COL_RESUMEN_INI + j).column_letter].width = w

    ws.freeze_panes = f"B{fila_labels + 1}"

    wr = wb.create_sheet("Resumen Final")
    resumen_final_headers = ["TRIP", "Fecha", "DíaSEM", "Vuelo", "Ruta", "BP INS",
                              "INS", "ACTIVIDAD", "FLOTA", "Nombre INS", "CUPOS"]
    for j, h in enumerate(resumen_final_headers):
        c = wr.cell(row=1, column=1 + j, value=h)
        c.font = bold
        c.border = border

    P = "'Pairings NB'!"
    for i, (r_ida, r_vta, r1_bloque) in enumerate(todas_filas_pairing, start=2):
        ins_ref = f"{P}$M${r1_bloque}"
        act_ref = f"{P}$O${r1_bloque}"
        flota_ref = f"{P}J{r_ida}"
        valores = [
            f"={P}B{r_ida}",
            f"={P}C{r_ida}",
            f"={P}D{r_ida}",
            f'={P}E{r_ida}&"/"&{P}E{r_vta}',
            f'={P}F{r_ida}&"-"&{P}G{r_ida}&"-"&{P}F{r_ida}',
            f'=IFERROR(VLOOKUP({ins_ref},Instructores!$A:$C,2,FALSE),"")',
            f"={ins_ref}",
            f"={act_ref}",
            f"={flota_ref}",
            f'=IFERROR(VLOOKUP({ins_ref},Instructores!$A:$C,3,FALSE),"")',
            f'=IF({flota_ref}=320,4,3)',
        ]
        for j, val in enumerate(valores):
            c = wr.cell(row=i, column=1 + j, value=val)
            c.border = border

    for col, w in zip("ABCDEFGHIJK", (9, 11, 10, 12, 16, 10, 18, 26, 8, 40, 8)):
        wr.column_dimensions[col].width = w
    wr.freeze_panes = "A2"
    wr.auto_filter.ref = wr.dimensions

    wb.save(out_path)


OUT = "Pairings_NB_con_instructores.xlsx"
construir_bloques_nb(bloques, OUT, n_muestra=None, asignaciones_por_bloque=asignaciones_por_bloque)
print(f"Archivo generado: {OUT}")
print(f"Bloques con Instructor+Actividad ya completados: {len(asignaciones_por_bloque)} de {len(bloques)} bloques totales")


## 6. QA: recalcular en Python que el llenado quedó bien (no solo mirar el Excel)

In [ ]:
nombres_catalogo = {n for n, _bp, _legal in INSTRUCTORES_DATA}
malos = [(i, n) for i, (n, _a) in asignaciones_por_bloque.items() if n not in nombres_catalogo]
print("Asignaciones con nombre fuera del catálogo:", len(malos))

print("Bloques asignados (deben ser todos únicos, por construcción del dict):", len(asignaciones_por_bloque))

inconsistencias_fecha = 0
for idx, (nombre, _act) in asignaciones_por_bloque.items():
    fecha_bloque = bloques[idx][0]["Fecha"]
    if not any(normalizar_fecha(f) == normalizar_fecha(fecha_bloque) for _bp, _nom, f in reservas):
        inconsistencias_fecha += 1
print("Bloques asignados cuya fecha no estaba reservada en la Matriz:", inconsistencias_fecha)

ids_usados = []
for a, b in bloques:
    ids_usados.append(a["Pairing ID"])
    if b is not None:
        ids_usados.append(b["Pairing ID"])
rep = [pid for pid, c in collections.Counter(ids_usados).items() if c > 1]
print("Pairing IDs repetidos entre bloques (deben ser 0):", len(rep))


## 7. Descargar el archivo final

In [ ]:
from google.colab import files
files.download(OUT)


## 8. Estado y pendientes — sin inventar nada

### Lo que se automatiza en este notebook
- Lee la Matriz real (`Rol de Instructores`) y detecta todos los slots `"LCK A320F"` ya
  reservados por el notebook de Fase 1/2.
- Arma los mismos bloques candidatos de siempre (BigQuery + reglas 2.9-2.14 + `parear_candidatos`,
  sin ningún cambio).
- Empareja cada (instructor, fecha) reservado con un bloque completo de esa fecha exacta, y
  llena Instructor + Actividad ("LCK A320F") directamente en el Excel de bloques.
- Reporta aparte (sin inventar una solución) los casos sin bloque disponible esa fecha, y los
  nombres que no matchean el catálogo de Instructores.

### Lo que sigue sin automatizar
- **Control de pairing duplicado en la hoja "Vuelos" real** (paso 16 del documento): acá se
  verifica que no se repita un Pairing ID entre bloques generados en la MISMA corrida, pero si
  el archivo de Pairings real ya tenía otros LCK cargados de una corrida anterior o de otro
  proceso (auditorías, habilitación, etc.), este notebook no los conoce y no puede chequear
  contra ellos.
- **Otras actividades** (Auditoría CAB Vuelo, Habilitación, Bianual IDE, etc.) — siguen
  siendo 100% manuales, este notebook solo llena "LCK A320F".
- **PDR / descanso reglamentario exacto** contra vuelos reales de línea del instructor (fuera
  de los LCK) — no se calcula, igual que en los notebooks anteriores.
- Si `sin_bloque` o `sin_match_nombre` no están vacíos, esos casos necesitan ajuste manual
  antes de dar el archivo por definitivo (mover el slot en la Matriz, o revisar por qué no
  hay un bloque candidato válido esa fecha).
